In [0]:
from pyspark.sql.functions import *

In [0]:
data = [
    (1, "HR", 5000),
    (2, "Engineering", 7000),
    (3, "HR", 6000),
    (4, "Engineering", 8000),
    (5, "Marketing", 4500),
    (6, "Engineering", 9000),
]

columns = ["id", "dept", "salary"]

# 1. Create source DataFrame
df = spark.createDataFrame(data, columns)

In [0]:
display(df)

In [0]:
df_agg = df.groupBy("dept").agg(avg("salary").alias("avg_salary"),count("id").alias("employee_count"))

In [0]:
display(df_agg)

##### Find Top N Customers
- Finding top customers is useful for business analytics and performance tracking.
- Aggregation calculates total spending per customer, and orderBy() sorts results in
descending order. limit() selects the top N customers efficiently. 
- This technique is widely used in financial and marketing analytics.

In [0]:
data = [
    (101, "Alice", 250.0),
    (102, "Bob", 150.0),
    (101, "Alice", 300.0),      # Alice total: 550.0
    (103, "Charlie", 700.0),    # Charlie total: 700.0
    (104, "Diana", 120.0),      # Diana total: 120.0
    (102, "Bob", 450.0),        # Bob total: 600.0
    (105, "Evan", 80.0),        # Evan total: 80.0
]

columns = ["customer_id", "customer_name", "amount"]

# 1. Create source DataFrame
df = spark.createDataFrame(data, columns)

In [0]:
display(df)

In [0]:
df_top_n = (df.groupBy("customer_id").agg(sum("amount").alias("total_per_customer")).orderBy(col("total_per_customer").desc()).limit(5))

In [0]:
display(df_top_n)

##### Running Totals over Time
- Running totals calculate cumulative values over time for analysis and reporting.
- Window functions enable aggregation without collapsing rows. partitionBy() groups records, and orderBy() defines sequence. 
- This is useful in financial reporting and trend analysis.

In [0]:
from pyspark.sql.functions import *

data = [
    ("Alice", "2024-01-01", 100),
    ("Alice", "2024-01-05", 150),
    ("Alice", "2024-01-05", 250),
    ("Alice", "2024-01-12", 200),
    ("Bob", "2024-01-02", 300),
    ("Bob", "2024-01-10", 120),
    ("Charlie", "2024-01-04", 500),
]

columns = ["customer", "date", "amount"]

# 1. Create source DataFrame
df = spark.createDataFrame(data, columns).withColumn("date", col("date").cast("date"))

In [0]:
display(df)

In [0]:
from pyspark.sql.window import Window

#window_spec = Window.partitionBy("customer").orderBy(col("date").asc()).rowsBetween(Window.unboundedPreceding, Window.currentRow)

#new Learning on default rowsBetween
window_spec = Window.partitionBy("customer").orderBy(col("date").asc()).rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_running = df.withColumn("running_total", sum("amount").over(window_spec))

In [0]:
display(df_running)

##### Moving Averages
- Moving averages smooth data and help identify trends over time. 
- Window functions allow calculating averages across a defined range of rows. rowsBetween() defines sliding window boundaries. 
- **This technique is widely used in time-series analytics and forecasting.**

In [0]:
data = [
    ("Alice", "2024-01-01", 100.0),
    ("Alice", "2024-01-02", 200.0),
    ("Alice", "2024-01-03", 300.0),
    ("Alice", "2024-01-04", 400.0),
    ("Bob", "2024-01-01", 50.0),
    ("Bob", "2024-01-02", 150.0),
    ("Bob", "2024-01-03", 100.0),
]

columns = ["customer", "date", "amount"]

# 1. Create source DataFrame
df = spark.createDataFrame(data, columns).withColumn("date", col("date").cast("date"))

In [0]:
display(df)

In [0]:
window_spec_ma = Window.partitionBy("customer").orderBy("date").rowsBetween(-2,0)

df_moving_avg = df.withColumn("moving_avg", avg("amount").over(window_spec_ma))

In [0]:
display(df_moving_avg)

##### Transaction Count per Customer/Day
- Counting transactions per customer per day helps analyze customer behavior and
activity patterns. 
- The groupBy() function allows grouping by multiple columns such as date and customer ID. 
- The count() aggregation calculates the total number of transactions efficiently. 
- This technique is widely used in analytics dashboards and reporting pipelines.

In [0]:
data = [
    ("2026-08-01", "Alice", 101, 45.50),
    ("2026-08-01", "Alice", 102, 12.00),
    ("2026-08-01", "Bob",   103, 99.99),
    ("2026-08-02", "Alice", 104, 30.00),
    ("2026-08-02", "Bob",   105, 15.20),
    ("2026-08-02", "Bob",   106, 50.00),
    ("2026-08-02", "Charlie", 107, 75.00),
]

columns = ["date", "customer", "txn_id", "amount"]

# 3. Create DataFrame
df = spark.createDataFrame(data, schema=columns)

In [0]:
df_count = df.groupBy("customer","date").agg(count("*").alias("transaction_count"))

In [0]:
display(df_count)

##### Identify Employees Without Departments
- **Left anti join** identifies records present in one dataset but missing in another. 
- It is useful for finding orphan records or missing relationships. This helps detect data quality issues and incomplete records. 
- It is widely used in data validation and reconciliation pipelines.

In [0]:
emp_data = [
    (1, "Alice", 10),
    (2, "Bob", 20),
    (3, "Charlie", 30),
    (4, "David", 40),   # Department 40 doesn't exist
    (5, "Eve", None)     # No department assigned
]
emp_schema = ["emp_id", "emp_name", "dept_id"]
df_emp = spark.createDataFrame(emp_data, schema=emp_schema)

# 2. Department data
dept_data = [
    (10, "Engineering"),
    (20, "Marketing"),
    (30, "HR")
]
dept_schema = ["dept_id", "dept_name"]
df_dept = spark.createDataFrame(dept_data, schema=dept_schema)

In [0]:
df_no_dept = df_emp.join(df_dept, on = df_emp.dept_id == df_dept.dept_id, how = "left_anti")
df_no_dept2 = df_emp.join(df_dept, on = df_emp.dept_id == df_dept.dept_id, how = "left").where(df_dept.dept_id.isNull())

In [0]:
display(df_no_dept)

In [0]:
display(df_no_dept2)

#### Find Products Never Sold
- This scenario identifies products with no matching transactions. A left join followed by filtering NULL values finds unmatched records. 
- This helps identify inactive products or inventory gaps. It is commonly used in retail analytics and inventory
management.

- left join followed by isNull check on product id, products on left and transactions on right
- left anti will also work

##### Highest-Paid Employee per Department
- Window functions allow ranking records within partitions. The row_number() function assigns rank based on salary order. Filtering rank = 1 returns highest-paid employee per department. 
- This is useful in HR analytics and reporting.

In [0]:
data = [
    (1, "Alice", "Engineering", 120000),
    (2, "Bob", "Engineering", 95000),
    (3, "Charlie", "Engineering", 110000),
    (4, "David", "Marketing", 85000),
    (5, "Eve", "Marketing", 92000),
    (6, "Frank", "HR", 75000),
    (7, "Grace", "HR", 78000),
]

schema = ["emp_id", "emp_name", "dept", "salary"]
df = spark.createDataFrame(data, schema=schema)

In [0]:
from pyspark.sql.window import Window

window_spec_emp = Window.partitionBy("dept").orderBy(col("salary").desc())

df_emp_hp = df.withColumn("rank",row_number().over(window_spec_emp)).filter("rank == 1").drop("rank")

In [0]:
display(df_emp_hp)

##### Rank Products by Revenue in Category
- Ranking allows identifying top-performing products within each category. The rank() window function assigns rankings based on revenue order. partitionBy() ensures ranking happens within each category independently. 
- This technique is widely used in business analytics to identify best-selling products and optimize strategy.

In [0]:
data = [
    ("Laptop", "Electronics", 120000),
    ("Smartphone", "Electronics", 120000),  # Tied for 1st
    ("Tablet", "Electronics", 45000),
    ("Headphones", "Electronics", 20000),
    ("Sofa", "Furniture", 85000),
    ("Dining Table", "Furniture", 60000),    # Tied for 2nd
    ("Office Chair", "Furniture", 60000),    # Tied for 2nd
    ("Bookshelf", "Furniture", 30000),
]

schema = ["product", "category", "revenue"]

# 3. Create source DataFrame
df = spark.createDataFrame(data, schema=schema)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

window_spec = Window.partitionBy("category").orderBy(col("revenue").desc())

df_rank = df.withColumn("rank", rank().over(window_spec))

In [0]:
display(df_rank)

##### Find the Second Highest Salary
- Window functions enable ranking records globally without aggregation. The rank() function assigns order based on salary. Filtering rank = 2 retrieves the second highest salary. 
- This approach is efficient and commonly asked in interviews.

In [0]:
data = [
    (1, "Alice",   120000),
    (2, "Alisha",   120000),  # Rank 1
    (3, "Bob",     110000),  # Rank 2 (Tie)
    (4, "Charlie", 110000),  # Rank 2 (Tie)
    (5, "David",   95000),   # Rank 4 (skipped 3 due to F.rank)
    (6, "Eve",     80000),   # Rank 5
]

schema = ["emp_id", "emp_name", "salary"]

df = spark.createDataFrame(data, schema=schema)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *


window_spec = Window.orderBy(col("salary").desc())

df_2_salary = df.withColumn("rank",dense_rank().over(window_spec)).filter("rank == 2").drop("rank")

In [0]:
display(df_2_salary)

##### Deduplicate Latest Record per ID
- Deduplication ensures only the latest record is retained per ID. row_number() assigns rank based on timestamp order. 
- Filtering row_number = 1 keeps latest record. This is widely used in CDC pipelines.

In [0]:
data = [
    (101, "Alice",   "Active",    "2026-01-10"),
    (101, "Alice",   "Suspended", "2026-03-15"),
    (101, "Alice",   "Active",    "2026-05-20"),  # Latest for 101
    (102, "Bob",     "Pending",   "2026-02-01"),
    (102, "Bob",     "Active",    "2026-04-11"),  # Latest for 102
    (103, "Charlie", "Active",    "2026-01-05"),  # Latest for 103
]

schema = ["id", "name", "status", "update_date"]

# 3. Create source DataFrame
df = spark.createDataFrame(data, schema=schema)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

window_spec = Window.partitionBy("id").orderBy(col("update_date").desc())

df_latest = df.withColumn("rank",row_number().over(window_spec))

In [0]:
display(df_latest.filter("rank == 1").drop("rank"))

##### Identify Changes using MD5 Hashing
- Hashing allows efficient comparison between datasets. MD5 generates unique hash values based on column data. Comparing hashes identifies changed records. 
- This technique is widely used in incremental data pipelines.

In [0]:
from pyspark.sql import functions as F

staging_data = [
    (1, "Alice", 95000),    # Unchanged
    (2, "Bob", 115000),     # Updated (salary changed)
    (3, "Charlie", 120000)  # Insert (new employee)
]
staging_schema = ["emp_id", "name", "salary"]
df_staging = spark.createDataFrame(staging_data, schema=staging_schema)

In [0]:
df_staging_hashed = df_staging.withColumn("payload_hash", F.md5(F.concat_ws("||",F.col("name"),F.col("salary"))))

In [0]:
display(df_staging_hashed)

In [0]:
target_base_data = [
    (1, "Alice", 95000),    # Matches staging
    (2, "Bob", 110000),     # Old salary
    (4, "David", 85000)     # Exists only in target
]
target_schema = ["emp_id", "name", "salary"]

# Generate existing target table with stored payload_hash
df_target = spark.createDataFrame(target_base_data, schema=target_schema).withColumn("payload_hash", F.md5(F.concat_ws("||", F.col("name"), F.col("salary"))))

In [0]:
df_changes = (df_staging_hashed.alias("stg")
              .join(df_target.alias("tgt"),"emp_id","left")
              .filter(
                      (F.col("tgt.payload_hash").isNull()) | 
                      (F.col("stg.payload_hash") != F.col("tgt.payload_hash"))
                      )
              )

In [0]:
display(df_staging_hashed.alias("stg").join(df_target.alias("tgt"),"emp_id","left"))

In [0]:
display(df_changes)

##### CDC Classification (NEW/UPDATED/DELETED)
- CDC identifies changes between source and target datasets. Full outer join compares records from both datasets. 
- Conditional logic classifies changes as new, updated, or deleted. CDC pipelines are critical in data warehousing.

In [0]:
src_data = [
    (1, "hash_a_v1"),  # UNCHANGED (matches target)
    (2, "hash_b_v2"),  # UPDATED (target has hash_b_v1)
    (3, "hash_c_v1"),  # NEW (not in target)
]

tgt_data = [
    (1, "hash_a_v1"),  # UNCHANGED
    (2, "hash_b_v1"),  # UPDATED
    (4, "hash_d_v1"),  # DELETED (not in source)
]

df_src = spark.createDataFrame(src_data, ["emp_id", "payload_hash"])
df_tgt = spark.createDataFrame(tgt_data, ["emp_id", "payload_hash"])

In [0]:
from pyspark.sql import functions as F

df_joined = (df_src.alias("src").join(df_tgt.alias("tgt"),
                "emp_id",
                "full_outer"
            ))

In [0]:
df_cdc = df_joined.withColumn("change_type", 
                              F.when(F.col("src.emp_id").isNotNull() & F.col("tgt.emp_id").isNull(), "NEW")
                              .when(F.col("src.payload_hash") !=F.col("tgt.payload_hash"),"UPDATED")
                              .when(F.col("src.emp_id").isNull(),"DELETED")
                              .otherwise("UNCHANGED")
                              )

In [0]:
display(df_joined)

In [0]:
display(df_cdc)